In [1]:
#caso con ADAF+SE soltanto


import numpy as np
import pandas as pd


# =============================================================================
# COSTANTI FISICHE
# =============================================================================

c = 2.99792458e10
G = 6.67430e-8
h = 6.62607015e-27
kB = 1.380649e-16
sigma = 5.670374419e-5
Msun = 1.98847e33


# =============================================================================
# FILE DI INPUT
# =============================================================================

file_linee = "Tab_completa_revisione.csv"
file_transizioni = "dati.asc"


# =============================================================================
# J0508 - PARAMETRI OTTENUTI DAL NUOVO MCMC ADAF + SE
# =============================================================================

# Inserire qui i valori mediani ottenuti con il nuovo MCMC

logM = 6.524
logmdot = 0.383


MBH_solar = 10**logM
mdot = 10**logmdot


# =============================================================================
# UNICO SWITCH TRA ADAF E RAMO RADIATIVAMENTE EFFICIENTE
#
# log10(mdot) < -2  -> ADAF
# log10(mdot) >= -2 -> ramo SE
#
# Nel ramo SE:
#
# F_disk(R) = min[F_SS(R), F_Edd(R)]
#
# quindi:
#
# - se F_SS <= F_Edd ovunque -> spettro SS-like
# - se F_SS > F_Edd in qualche regione -> spettro SE-capped
# =============================================================================

logmdot_switch_adaf_se = -2.0


if logmdot < logmdot_switch_adaf_se:

    regime = "ADAF"

else:

    regime = "SE"


print()
print("=" * 100)
print("J0508 - ACCRETION MODEL AND COVERING FACTOR")
print("=" * 100)

print(f"log10(M/Msun) = {logM:.6f}")
print(f"M             = {MBH_solar:.4e} Msun")

print(f"log10(mdot)   = {logmdot:.6f}")
print(f"mdot          = {mdot:.4e}")

print(f"MCMC branch   = {regime}")


# =============================================================================
# CONTROLLO
#
# Questo programma è quello specifico per J0508 e costruisce il ramo
# radiativamente efficiente SS/SE.
#
# Se in futuro i nuovi risultati MCMC portassero J0508 sotto mdot=1e-2,
# andrebbe invece utilizzato lo spettro ADAF.
# =============================================================================

if regime == "ADAF":

    raise ValueError(
        "J0508 risulta nel regime ADAF con i valori MCMC inseriti. "
        "Questo programma specifico deve quindi utilizzare il modello ADAF "
        "invece del ramo SS/SE."
    )


# =============================================================================
# PARAMETRI DEL DISCO
# =============================================================================

eta = 0.083

Nr = 4000
Nnu = 1000

nu_min = 1e13
nu_max = 1e17

nu_max_ion = 1e17


# =============================================================================
# MASSA DEL BUCO NERO IN CGS
# =============================================================================

MBH = (
    MBH_solar
    *
    Msun
)


# =============================================================================
# LUMINOSITA' DI EDDINGTON
# =============================================================================

Ledd = (
    1.3e38
    *
    MBH_solar
)


# =============================================================================
# TASSO DI ACCRESCIMENTO FISICO
# =============================================================================

# Stessa convenzione utilizzata nel MCMC:
#
# mdot = Ldisk_SS,input / Ledd
#
# quindi il valore mdot viene inizialmente interpretato come
# luminosità che avrebbe il disco SS senza il capping.

Ldisk_SS_input = (
    mdot
    *
    Ledd
)


Mdot = (
    Ldisk_SS_input
    /
    (
        eta
        *
        c**2
    )
)


# =============================================================================
# RAGGI DEL DISCO
# =============================================================================

Rs = (
    2
    *
    G
    *
    MBH
    /
    c**2
)


Rin = (
    3
    *
    Rs
)


Rout = (
    3000
    *
    Rs
)


# =============================================================================
# GRIGLIA RADIALE
# =============================================================================

R = np.logspace(
    np.log10(Rin),
    np.log10(Rout),
    Nr
)


# =============================================================================
# FATTORE DI BORDO SHAKURA-SUNYAEV
# =============================================================================

f = (
    1.0
    -
    np.sqrt(
        Rin
        /
        R
    )
)


f = np.clip(
    f,
    0.0,
    None
)


# =============================================================================
# FLUSSO SHAKURA-SUNYAEV
# =============================================================================

F_SS = (

    3
    *
    G
    *
    MBH
    *
    Mdot

    /

    (
        8
        *
        np.pi
        *
        R**3
    )

    *

    f
)


# =============================================================================
# FLUSSO LOCALE DI EDDINGTON
# =============================================================================

F_Edd = (

    Ledd

    /

    (
        4
        *
        np.pi
        *
        R**2
    )
)


# =============================================================================
# RAMO RADIATIVAMENTE EFFICIENTE SS / SE
# =============================================================================

# Non esiste più uno switch separato SS -> SE.
#
# Si confrontano direttamente i due flussi locali:
#
# F_SS <= F_Edd  -> F_disk = F_SS
#
# F_SS > F_Edd   -> F_disk = F_Edd
#
# cioè:
#
# F_disk = min(F_SS, F_Edd)

F_disk = np.minimum(
    F_SS,
    F_Edd
)


# =============================================================================
# REGIONE IN CUI IL CAPPING E' ATTIVO
# =============================================================================

mask_Edd = (
    F_SS
    >
    F_Edd
)


has_eddington_limited_region = (
    bool(
        np.any(
            mask_Edd
        )
    )
)


if has_eddington_limited_region:

    effective_disk_state = "SE-capped"

    indices_Edd = np.where(
        mask_Edd
    )[0]

    R_Edd_inner = (
        R[
            indices_Edd[0]
        ]
    )

    R_Edd_outer = (
        R[
            indices_Edd[-1]
        ]
    )

else:

    effective_disk_state = "SS-like"

    R_Edd_inner = None
    R_Edd_outer = None


# =============================================================================
# TEMPERATURA DEL DISCO
# =============================================================================

# sigma T^4 = F_disk

T_disk = (
    F_disk
    /
    sigma
)**0.25


# =============================================================================
# GRIGLIA DI FREQUENZA
# =============================================================================

nu = np.logspace(
    np.log10(nu_min),
    np.log10(nu_max),
    Nnu
)


# =============================================================================
# FUNZIONE DI PLANCK
# =============================================================================

Bnu = np.zeros(
    (
        Nnu,
        Nr
    )
)


for j in range(Nr):

    Tj = (
        T_disk[j]
    )

    if Tj <= 0:

        continue


    x = (

        h
        *
        nu

        /

        (
            kB
            *
            Tj
        )
    )


    x = np.clip(
        x,
        1e-12,
        700.0
    )


    Bnu[:, j] = (

        2
        *
        h
        *
        nu**3

        /

        c**2

    ) / np.expm1(x)


# =============================================================================
# CALCOLO DI L_nu
# =============================================================================

integrand = (
    R[None, :]
    *
    Bnu
)


I_nu = np.trapezoid(
    integrand,
    R,
    axis=1
)


Lnu = (
    4
    *
    np.pi**2
    *
    I_nu
)


nuLnu = (
    nu
    *
    Lnu
)


# =============================================================================
# LUMINOSITA' DEL DISCO
# =============================================================================

# Entrambe le facce del disco:
#
# dL = 4 pi R F(R) dR

Ldisk_model = (

    4
    *
    np.pi

    *

    np.trapezoid(
        R
        *
        F_disk,
        R
    )
)


# =============================================================================
# INFORMAZIONI SUL MODELLO
# =============================================================================

print()
print("-" * 100)
print("RADIATIVELY EFFICIENT DISK BRANCH")
print("-" * 100)

print(
    f"MCMC branch        = "
    f"{regime}"
)

print(
    f"Effective state    = "
    f"{effective_disk_state}"
)

print(
    f"Eddington cap      = "
    f"{has_eddington_limited_region}"
)

print(
    f"Ledd               = "
    f"{Ledd:.4e} erg/s"
)

print(
    f"Ldisk SS input     = "
    f"{Ldisk_SS_input:.4e} erg/s"
)

print(
    f"Ldisk model        = "
    f"{Ldisk_model:.4e} erg/s"
)

print(
    f"Ldisk model/Ledd   = "
    f"{Ldisk_model/Ledd:.4f}"
)

print(
    f"Tmax               = "
    f"{np.max(T_disk):.4e} K"
)


if has_eddington_limited_region:

    print(
        f"R_Edd_inner        = "
        f"{R_Edd_inner/Rs:.4f} Rs"
    )

    print(
        f"R_Edd_outer        = "
        f"{R_Edd_outer/Rs:.4f} Rs"
    )

else:

    print(
        "No radial region reaches the local Eddington flux."
    )


# =============================================================================
# LETTURA DEL FILE DELLE LUMINOSITA' OSSERVATE
# =============================================================================

df_lines = pd.read_csv(
    file_linee
)


# =============================================================================
# TROVO AUTOMATICAMENTE LA COLONNA DEL NOME DEL BLAZAR
# =============================================================================

possible_source_columns = [
    "label",
    "file_name",
    "name",
    "nome",
    "blazar"
]


source_column = None


for column in possible_source_columns:

    if column in df_lines.columns:

        source_column = column

        break


if source_column is None:

    raise ValueError(
        "Non trovo una colonna contenente il nome del blazar."
    )


# =============================================================================
# SELEZIONO J0508
# =============================================================================

mask_J0508 = (

    df_lines[
        source_column
    ]

    .astype(str)

    .str.contains(
        "J0508",
        case=False,
        na=False
    )
)


df_J0508 = (

    df_lines[
        mask_J0508
    ]

    .copy()
)


if len(df_J0508) == 0:

    raise ValueError(
        "J0508 non trovato in Tab_completa_revisione.csv"
    )


if len(df_J0508) > 1:

    print()
    print(
        "ATTENZIONE: trovate piu' sorgenti compatibili con J0508:"
    )

    print(
        df_J0508[
            source_column
        ].tolist()
    )

    print(
        "Uso la prima."
    )


row_J0508 = (
    df_J0508.iloc[0]
)


print()
print(
    f"Sorgente selezionata: "
    f"{row_J0508[source_column]}"
)


# =============================================================================
# LETTURA DEL FILE DELLE TRANSIZIONI
# =============================================================================

(
    line_names,
    ionization_energies,
    ionization_frequencies

) = np.genfromtxt(

    file_transizioni,

    dtype=(
        "U50",
        float,
        float
    ),

    unpack=True
)


df_transitions = pd.DataFrame({

    "line_name":
        line_names,

    "E_ion":
        ionization_energies,

    "nu_ion":
        ionization_frequencies
})


# =============================================================================
# RIGHE CHE VOGLIAMO STUDIARE
# =============================================================================

lines_to_use = [
    "CIII",
    "CIV"
]


# =============================================================================
# CALCOLO DI L_ION E COVERING FACTOR
# =============================================================================

results = []


for line_name in lines_to_use:


    # =========================================================================
    # FREQUENZA DI IONIZZAZIONE
    # =========================================================================

    transition_match = (

        df_transitions[

            df_transitions[
                "line_name"
            ]

            ==

            line_name
        ]
    )


    if len(transition_match) == 0:

        print(
            f"{line_name}: transizione non trovata in dati.asc"
        )

        continue


    transition_row = (
        transition_match.iloc[0]
    )


    E_ion = (
        transition_row[
            "E_ion"
        ]
    )


    nu_ion = (
        transition_row[
            "nu_ion"
        ]
    )


    # =========================================================================
    # LUMINOSITA' OSSERVATA
    # =========================================================================

    norm_column = (
        f"{line_name}_norm"
    )


    uncertainty_column = (
        f"{line_name}_norm_unc"
    )


    if norm_column not in row_J0508.index:

        print(
            f"{line_name}: colonna {norm_column} non trovata."
        )

        continue


    norm_value = (
        row_J0508[
            norm_column
        ]
    )


    if (
        pd.isna(norm_value)
        or
        norm_value <= 0
    ):

        print(
            f"{line_name}: luminosita' osservata non valida."
        )

        continue


    # stessa conversione usata nel programma MCMC:
    #
    # valori *_norm espressi in 10^42 erg/s

    L_obs = (
        norm_value
        *
        1e42
    )


    # =========================================================================
    # INCERTEZZA OSSERVATIVA
    # =========================================================================

    if (
        uncertainty_column
        in row_J0508.index

        and

        pd.notna(
            row_J0508[
                uncertainty_column
            ]
        )

        and

        row_J0508[
            uncertainty_column
        ] > 0
    ):

        sigma_L_obs = (

            row_J0508[
                uncertainty_column
            ]

            *

            1e42
        )


    else:

        sigma_L_obs = np.nan


    # =========================================================================
    # LUMINOSITA' IONIZZANTE
    # =========================================================================

    # Per CIII e CIV:
    #
    #
    #                     nu_max
    # L_ion = integral      L_nu dnu
    #                     nu_ion
    #

    ionizing_mask = (

        (nu >= nu_ion)

        &

        (nu <= nu_max_ion)
    )


    if np.sum(
        ionizing_mask
    ) < 2:

        L_ion = np.nan


    else:

        L_ion = np.trapezoid(

            Lnu[
                ionizing_mask
            ],

            nu[
                ionizing_mask
            ]
        )


    # =========================================================================
    # COVERING FACTOR
    # =========================================================================

    # Modello:
    #
    #
    # L_line = Cf * L_ion
    #
    #
    # quindi:
    #
    #
    #            L_obs
    # Cf = -----------------
    #            L_ion
    #

    if (
        np.isfinite(L_ion)
        and
        L_ion > 0
    ):

        Cf = (
            L_obs
            /
            L_ion
        )


    else:

        Cf = np.nan


    # =========================================================================
    # INCERTEZZA SU Cf
    # =========================================================================

    # Qui propaghiamo solamente l'incertezza osservativa
    # sulla luminosita' della riga.
    #
    # Non includiamo l'incertezza MCMC su M e mdot.

    if (
        np.isfinite(Cf)
        and
        np.isfinite(sigma_L_obs)
        and
        L_ion > 0
    ):

        sigma_Cf = (

            sigma_L_obs

            /

            L_ion
        )


    else:

        sigma_Cf = np.nan


    # =========================================================================
    # SALVO RISULTATI
    # =========================================================================

    results.append({

        "line_name":
            line_name,

        "E_ion_eV":
            E_ion,

        "nu_ion_Hz":
            nu_ion,

        "L_obs_erg_s":
            L_obs,

        "sigma_L_obs_erg_s":
            sigma_L_obs,

        "L_ion_erg_s":
            L_ion,

        "Cf":
            Cf,

        "sigma_Cf":
            sigma_Cf,

        "Cf_percent":
            100 * Cf,

        "sigma_Cf_percent":
            100 * sigma_Cf
    })


# =============================================================================
# DATAFRAME FINALE
# =============================================================================

df_results = pd.DataFrame(
    results
)


# =============================================================================
# STAMPA DEI RISULTATI
# =============================================================================

print()
print("=" * 100)
print("COVERING FACTOR DI J0508")
print("=" * 100)


for _, row in df_results.iterrows():

    print()

    print(
        f"LINEA = {row['line_name']}"
    )

    print(
        f"E_ion        = "
        f"{row['E_ion_eV']:.4f} eV"
    )

    print(
        f"nu_ion       = "
        f"{row['nu_ion_Hz']:.4e} Hz"
    )

    print(
        f"L_obs        = "
        f"{row['L_obs_erg_s']:.4e} erg/s"
    )

    print(
        f"L_ion        = "
        f"{row['L_ion_erg_s']:.4e} erg/s"
    )

    print(
        f"Cf           = "
        f"{row['Cf']:.4f}"
    )

    print(
        f"Cf percent   = "
        f"{row['Cf_percent']:.2f} %"
    )

    if np.isfinite(
        row[
            "sigma_Cf"
        ]
    ):

        print(
            f"sigma Cf     = "
            f"{row['sigma_Cf']:.4f}"
        )

        print(
            f"sigma Cf (%) = "
            f"{row['sigma_Cf_percent']:.2f} %"
        )

    print("-" * 100)


print()
print(df_results)


J0508 - ACCRETION MODEL AND COVERING FACTOR
log10(M/Msun) = 6.524000
M             = 3.3420e+06 Msun
log10(mdot)   = 0.383000
mdot          = 2.4155e+00
MCMC branch   = SE

----------------------------------------------------------------------------------------------------
RADIATIVELY EFFICIENT DISK BRANCH
----------------------------------------------------------------------------------------------------
MCMC branch        = SE
Effective state    = SE-capped
Eddington cap      = True
Ledd               = 4.3445e+44 erg/s
Ldisk SS input     = 1.0494e+45 erg/s
Ldisk model        = 1.0363e+45 erg/s
Ldisk model/Ledd   = 2.3854
Tmax               = 4.1147e+05 K
R_Edd_inner        = 5.1248 Rs
R_Edd_outer        = 9.6605 Rs

Sorgente selezionata: J0508.2-1936

COVERING FACTOR DI J0508

LINEA = CIII
E_ion        = 24.3800 eV
nu_ion       = 5.8900e+15 Hz
L_obs        = 7.0353e+43 erg/s
L_ion        = 8.5599e+44 erg/s
Cf           = 0.0822
Cf percent   = 8.22 %
sigma Cf     = 0.0030
sigma Cf (

In [2]:
import numpy as np
import pandas as pd


# =============================================================================
# COSTANTI FISICHE
# =============================================================================

c = 2.99792458e10          # velocita' della luce [cm/s]
G = 6.67430e-8             # costante gravitazionale [cm^3 g^-1 s^-2]
h = 6.62607015e-27         # costante di Planck [erg s]
kB = 1.380649e-16          # costante di Boltzmann [erg/K]
sigma = 5.670374419e-5     # Stefan-Boltzmann [erg cm^-2 s^-1 K^-4]
Msun = 1.98847e33          # massa solare [g]


# =============================================================================
# FILE DI INPUT
# =============================================================================

file_linee = "Tab_completa_revisione.csv"
file_transizioni = "dati.asc"


# =============================================================================
# J0508 - PARAMETRI OTTENUTI DAL MCMC
# =============================================================================

# Valori mediani del posterior MCMC

logM = 6.487390
logmdot = 0.435499


# conversione dai logaritmi

MBH_solar = 10**logM          # massa del BH [Msun]
mdot = 10**logmdot            # tasso di accrescimento adimensionale


print()
print("=" * 100)
print("J0508 - MODELLO SUPER-EDDINGTON E COVERING FACTOR")
print("=" * 100)

print(f"log10(M/Msun) = {logM:.6f}")
print(f"M             = {MBH_solar:.4e} Msun")

print(f"log10(mdot)   = {logmdot:.6f}")
print(f"mdot          = {mdot:.4e}")

print("Regime        = SE")


# =============================================================================
# PARAMETRI DEL DISCO
# =============================================================================

eta = 0.083

Nr = 4000

Nnu = 1000

nu_min = 1e13
nu_max = 1e17

nu_max_ion = 1e17


# =============================================================================
# MASSA DEL BUCO NERO IN CGS
# =============================================================================

MBH = (
    MBH_solar
    *
    Msun
)


# =============================================================================
# LUMINOSITA' DI EDDINGTON
# =============================================================================

Ledd = (
    1.3e38
    *
    MBH_solar
)


# =============================================================================
# TASSO DI ACCRESCIMENTO FISICO
# =============================================================================

# Stessa convenzione utilizzata nel modello SS / SE:
#
# mdot = Ldisk_SS / Ledd
#
# quindi:
#
# Ldisk_SS_input = mdot * Ledd

Ldisk_SS_input = (
    mdot
    *
    Ledd
)


# tasso di accrescimento fisico

Mdot = (
    Ldisk_SS_input
    /
    (
        eta
        *
        c**2
    )
)


# =============================================================================
# RAGGI DEL DISCO
# =============================================================================

Rs = (
    2
    *
    G
    *
    MBH
    /
    c**2
)


Rin = (
    3
    *
    Rs
)


Rout = (
    3000
    *
    Rs
)


# =============================================================================
# GRIGLIA RADIALE
# =============================================================================

R = np.logspace(
    np.log10(Rin),
    np.log10(Rout),
    Nr
)


# =============================================================================
# FATTORE DI BORDO SHAKURA-SUNYAEV
# =============================================================================

f = (
    1.0
    -
    np.sqrt(
        Rin
        /
        R
    )
)


f = np.clip(
    f,
    0.0,
    None
)


# =============================================================================
# FLUSSO SHAKURA-SUNYAEV
# =============================================================================

# Flusso emesso da una superficie del disco:
#
#
#             3 G M Mdot
# F_SS(R) = -------------- [1 - sqrt(Rin/R)]
#              8 pi R^3
#

F_SS = (

    3
    *
    G
    *
    MBH
    *
    Mdot

    /

    (
        8
        *
        np.pi
        *
        R**3
    )

    *

    f
)


# =============================================================================
# FLUSSO LOCALE DI EDDINGTON
# =============================================================================

# Prescrizione SE semplificata:
#
#
#                  Ledd
# F_Edd(R) = ----------------
#               4 pi R^2
#

F_Edd = (

    Ledd

    /

    (
        4
        *
        np.pi
        *
        R**2
    )
)


# =============================================================================
# MODELLO SUPER-EDDINGTON
# =============================================================================

# Quando il normale flusso SS supera il limite locale di Eddington,
# il flusso viene saturato:
#
#
# F_SE = min(F_SS, F_Edd)
#

F_SE = np.minimum(
    F_SS,
    F_Edd
)


# =============================================================================
# TEMPERATURA DEL DISCO SE
# =============================================================================

# sigma T^4 = F
#
# quindi:
#
# T = (F/sigma)^(1/4)

T_SE = (

    F_SE

    /

    sigma

)**0.25


# =============================================================================
# REGIONE LIMITATA DAL FLUSSO DI EDDINGTON
# =============================================================================

mask_Edd = (
    F_SS
    >
    F_Edd
)


if np.any(mask_Edd):

    indices_Edd = np.where(
        mask_Edd
    )[0]

    R_Edd_inner = (
        R[
            indices_Edd[0]
        ]
    )

    R_Edd_outer = (
        R[
            indices_Edd[-1]
        ]
    )

else:

    R_Edd_inner = None
    R_Edd_outer = None


# =============================================================================
# GRIGLIA DI FREQUENZA
# =============================================================================

nu = np.logspace(
    np.log10(nu_min),
    np.log10(nu_max),
    Nnu
)


# =============================================================================
# FUNZIONE DI PLANCK
# =============================================================================

Bnu = np.zeros(
    (
        Nnu,
        Nr
    )
)


for j in range(Nr):

    Tj = (
        T_SE[j]
    )


    if Tj <= 0:

        continue


    x = (

        h
        *
        nu

        /

        (
            kB
            *
            Tj
        )
    )


    # protezione numerica

    x = np.clip(
        x,
        1e-12,
        700.0
    )


    Bnu[:, j] = (

        2
        *
        h
        *
        nu**3

        /

        c**2

    ) / np.expm1(x)


# =============================================================================
# CALCOLO DI L_nu
# =============================================================================

# Spettro multitemperatura:
#
#
# L_nu = 4 pi^2 integral R B_nu[T(R)] dR
#

integrand = (

    R[None, :]

    *

    Bnu
)


I_nu = np.trapezoid(

    integrand,

    R,

    axis=1
)


Lnu = (

    4

    *

    np.pi**2

    *

    I_nu
)


# =============================================================================
# nu L_nu
# =============================================================================

nuLnu = (
    nu
    *
    Lnu
)


# =============================================================================
# LUMINOSITA' BOLOMETRICA DEL DISCO SE
# =============================================================================

# considerando entrambe le facce:
#
#
# dL = 4 pi R F_SE(R) dR
#

Ldisk_SE = (

    4

    *

    np.pi

    *

    np.trapezoid(

        R
        *
        F_SE,

        R
    )
)


# =============================================================================
# INFORMAZIONI SUL MODELLO SE
# =============================================================================

print()
print("-" * 100)
print("MODELLO SE")
print("-" * 100)

print(
    f"Ledd              = "
    f"{Ledd:.4e} erg/s"
)

print(
    f"Ldisk SS input    = "
    f"{Ldisk_SS_input:.4e} erg/s"
)

print(
    f"Ldisk SE          = "
    f"{Ldisk_SE:.4e} erg/s"
)

print(
    f"Ldisk SE / Ledd   = "
    f"{Ldisk_SE/Ledd:.4f}"
)

print(
    f"Tmax SE           = "
    f"{np.max(T_SE):.4e} K"
)


if R_Edd_inner is not None:

    print(
        f"R_Edd_inner       = "
        f"{R_Edd_inner/Rs:.4f} Rs"
    )

    print(
        f"R_Edd_outer       = "
        f"{R_Edd_outer/Rs:.4f} Rs"
    )


# =============================================================================
# LETTURA DEL FILE DELLE LUMINOSITA' OSSERVATE
# =============================================================================

df_lines = pd.read_csv(
    file_linee
)


# =============================================================================
# TROVO AUTOMATICAMENTE LA COLONNA DEL NOME DEL BLAZAR
# =============================================================================

possible_source_columns = [
    "label",
    "file_name",
    "name",
    "nome",
    "blazar"
]


source_column = None


for column in possible_source_columns:

    if column in df_lines.columns:

        source_column = column

        break


if source_column is None:

    raise ValueError(
        "Non trovo una colonna contenente il nome del blazar."
    )


# =============================================================================
# SELEZIONO J0508
# =============================================================================

mask_J0508 = (

    df_lines[
        source_column
    ]

    .astype(str)

    .str.contains(
        "J0508",
        case=False,
        na=False
    )
)


df_J0508 = (

    df_lines[
        mask_J0508
    ]

    .copy()
)


if len(df_J0508) == 0:

    raise ValueError(
        "J0508 non trovato in Tab_completa_revisione.csv"
    )


if len(df_J0508) > 1:

    print()
    print(
        "ATTENZIONE: trovate piu' sorgenti compatibili con J0508:"
    )

    print(
        df_J0508[
            source_column
        ].tolist()
    )

    print(
        "Uso la prima."
    )


row_J0508 = (
    df_J0508.iloc[0]
)


print()
print(
    f"Sorgente selezionata: "
    f"{row_J0508[source_column]}"
)


# =============================================================================
# LETTURA DEL FILE DELLE TRANSIZIONI
# =============================================================================

(
    line_names,
    ionization_energies,
    ionization_frequencies

) = np.genfromtxt(

    file_transizioni,

    dtype=(
        "U50",
        float,
        float
    ),

    unpack=True
)


df_transitions = pd.DataFrame({

    "line_name":
        line_names,

    "E_ion":
        ionization_energies,

    "nu_ion":
        ionization_frequencies
})


# =============================================================================
# RIGHE CHE VOGLIAMO STUDIARE
# =============================================================================

lines_to_use = [
    "CIII",
    "CIV"
]


# =============================================================================
# CALCOLO DI L_ION E COVERING FACTOR
# =============================================================================

results = []


for line_name in lines_to_use:


    # =========================================================================
    # FREQUENZA DI IONIZZAZIONE
    # =========================================================================

    transition_match = (

        df_transitions[

            df_transitions[
                "line_name"
            ]

            ==

            line_name
        ]
    )


    if len(transition_match) == 0:

        print(
            f"{line_name}: transizione non trovata in dati.asc"
        )

        continue


    transition_row = (
        transition_match.iloc[0]
    )


    E_ion = (
        transition_row[
            "E_ion"
        ]
    )


    nu_ion = (
        transition_row[
            "nu_ion"
        ]
    )


    # =========================================================================
    # LUMINOSITA' OSSERVATA
    # =========================================================================

    norm_column = (
        f"{line_name}_norm"
    )


    uncertainty_column = (
        f"{line_name}_norm_unc"
    )


    if norm_column not in row_J0508.index:

        print(
            f"{line_name}: colonna {norm_column} non trovata."
        )

        continue


    norm_value = (
        row_J0508[
            norm_column
        ]
    )


    if (
        pd.isna(norm_value)
        or
        norm_value <= 0
    ):

        print(
            f"{line_name}: luminosita' osservata non valida."
        )

        continue


    # stessa conversione usata nel programma MCMC:
    #
    # valori *_norm espressi in 10^42 erg/s

    L_obs = (
        norm_value
        *
        1e42
    )


    # =========================================================================
    # INCERTEZZA OSSERVATIVA
    # =========================================================================

    if (
        uncertainty_column
        in row_J0508.index

        and

        pd.notna(
            row_J0508[
                uncertainty_column
            ]
        )

        and

        row_J0508[
            uncertainty_column
        ] > 0
    ):

        sigma_L_obs = (

            row_J0508[
                uncertainty_column
            ]

            *

            1e42
        )


    else:

        sigma_L_obs = np.nan


    # =========================================================================
    # LUMINOSITA' IONIZZANTE
    # =========================================================================

    # Per CIII e CIV:
    #
    #
    #                     nu_max
    # L_ion = integral      L_nu dnu
    #                     nu_ion
    #

    ionizing_mask = (

        (nu >= nu_ion)

        &

        (nu <= nu_max_ion)
    )


    if np.sum(
        ionizing_mask
    ) < 2:

        L_ion = np.nan


    else:

        L_ion = np.trapezoid(

            Lnu[
                ionizing_mask
            ],

            nu[
                ionizing_mask
            ]
        )


    # =========================================================================
    # COVERING FACTOR
    # =========================================================================

    # Modello:
    #
    #
    # L_line = Cf * L_ion
    #
    #
    # quindi:
    #
    #
    #            L_obs
    # Cf = -----------------
    #            L_ion
    #

    if (
        np.isfinite(L_ion)
        and
        L_ion > 0
    ):

        Cf = (
            L_obs
            /
            L_ion
        )


    else:

        Cf = np.nan


    # =========================================================================
    # INCERTEZZA SU Cf
    # =========================================================================

    # Qui propaghiamo solamente l'incertezza osservativa
    # sulla luminosita' della riga.
    #
    # Non includiamo l'incertezza MCMC su M e mdot.

    if (
        np.isfinite(Cf)
        and
        np.isfinite(sigma_L_obs)
        and
        L_ion > 0
    ):

        sigma_Cf = (

            sigma_L_obs

            /

            L_ion
        )


    else:

        sigma_Cf = np.nan


    # =========================================================================
    # SALVO RISULTATI
    # =========================================================================

    results.append({

        "line_name":
            line_name,

        "E_ion_eV":
            E_ion,

        "nu_ion_Hz":
            nu_ion,

        "L_obs_erg_s":
            L_obs,

        "sigma_L_obs_erg_s":
            sigma_L_obs,

        "L_ion_erg_s":
            L_ion,

        "Cf":
            Cf,

        "sigma_Cf":
            sigma_Cf,

        "Cf_percent":
            100 * Cf,

        "sigma_Cf_percent":
            100 * sigma_Cf
    })


# =============================================================================
# DATAFRAME FINALE
# =============================================================================

df_results = pd.DataFrame(
    results
)


# =============================================================================
# STAMPA DEI RISULTATI
# =============================================================================

print()
print("=" * 100)
print("COVERING FACTOR DI J0508")
print("=" * 100)


for _, row in df_results.iterrows():

    print()

    print(
        f"LINEA = {row['line_name']}"
    )

    print(
        f"E_ion        = "
        f"{row['E_ion_eV']:.4f} eV"
    )

    print(
        f"nu_ion       = "
        f"{row['nu_ion_Hz']:.4e} Hz"
    )

    print(
        f"L_obs        = "
        f"{row['L_obs_erg_s']:.4e} erg/s"
    )

    print(
        f"L_ion        = "
        f"{row['L_ion_erg_s']:.4e} erg/s"
    )

    print(
        f"Cf           = "
        f"{row['Cf']:.4f}"
    )

    print(
        f"Cf percent   = "
        f"{row['Cf_percent']:.2f} %"
    )

    if np.isfinite(
        row[
            "sigma_Cf"
        ]
    ):

        print(
            f"sigma Cf     = "
            f"{row['sigma_Cf']:.4f}"
        )

        print(
            f"sigma Cf (%) = "
            f"{row['sigma_Cf_percent']:.2f} %"
        )

    print("-" * 100)


print()
print(df_results)


J0508 - MODELLO SUPER-EDDINGTON E COVERING FACTOR
log10(M/Msun) = 6.487390
M             = 3.0718e+06 Msun
log10(mdot)   = 0.435499
mdot          = 2.7258e+00
Regime        = SE

----------------------------------------------------------------------------------------------------
MODELLO SE
----------------------------------------------------------------------------------------------------
Ledd              = 3.9933e+44 erg/s
Ldisk SS input    = 1.0885e+45 erg/s
Ldisk SE          = 1.0315e+45 erg/s
Ldisk SE / Ledd   = 2.5832
Tmax SE           = 4.3312e+05 K
R_Edd_inner       = 4.4866 Rs
R_Edd_outer       = 12.6045 Rs

Sorgente selezionata: J0508.2-1936

COVERING FACTOR DI J0508

LINEA = CIII
E_ion        = 24.3800 eV
nu_ion       = 5.8900e+15 Hz
L_obs        = 7.0353e+43 erg/s
L_ion        = 8.5579e+44 erg/s
Cf           = 0.0822
Cf percent   = 8.22 %
sigma Cf     = 0.0030
sigma Cf (%) = 0.30 %
--------------------------------------------------------------------------------------------